# 03 · Export

Confirms the chosen model after quantization, measures its speed, sets the shipped threshold, and writes everything both demos need. Design: `plans/03_export.md`.

- **Exports EfficientNet-B0**, the chosen model. Add `'mnv3s'` to `MODELS` to also convert MobileNetV3-Small for comparison.
- **Reads** (Drive): `data/manifest.csv`, `dataset.zip`, and notebook 02's models and results in `models/crop_lr1e-4/` and `results/crop_lr1e-4/`.
- **Writes** (Drive):
  - **`models/export/`:** the shipped `.tflite` (used by both the Android app and the web demo), all variants, `labels.txt`, `metadata.json`, `preprocess.py`, `preprocess.js` and `test_vectors/`.
  - **`results/crop_lr1e-4/`:** `export.md` and `export_runs.csv`.
- **Afterwards:** download `models/export/` from Drive into this project's `models/export/`. The kernel can't write to your laptop.

**Rules.** Seed, quantization variant and threshold are chosen on **val** only. Test scores are printed after the choice, for reporting.

**Run:** connect a Colab kernel (GPU helps but isn't needed), then run all cells.

In [2]:
import csv, json, math, random, shutil, subprocess, time, zipfile, collections, importlib.util, datetime as dt
from pathlib import Path
import numpy as np

# --- Settings -------------------------------------------------------------------
SOURCE_VARIANT = 'crop_lr1e-4'   # notebook 02 folder with the frozen settings
MODELS = ['effb0']               # the chosen model; add 'mnv3s' to also convert the MobileNet fallback
SHIP_MODEL = 'effb0'
SHIP_VARIANT = 'float16'         # decided 2026-11-27: same val scores as float32, half the size. CPU timing on free
                                 # Colab is too noisy to rank float16 vs float32 (they swapped between runs). None = rule only
TOLERANCE = 0.01                 # a variant may lose at most ~1 point of val acc and val damaged recall
SPEED_LIMIT_MS = 150             # Colab-CPU ms/image: slower than this -> warning (and fallback, if in MODELS)
RECALL_TARGET = 0.95             # shipped threshold: largest one with >= 95% damaged recall on val
N_REPRESENTATIVE = 200           # train images used to calibrate full-int8 quantization
TIMING_WARMUP, TIMING_RUNS = 5, 50
RUN_WEB_CHECK = True             # Node runs preprocess.js on the test vectors; its output goes through the shipped .tflite
EXPECTED_MANIFEST_ROWS = 1123

# --- Where things live (same as notebooks 01-02) ---------------------------------
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/seventx_prototype')  # change if you put the folder elsewhere
    ROOT = Path('/content/seventx')
    WORK = Path('/content/export_work')                         # big temporary files stay off Drive
    if not (ROOT / 'dataset').exists():
        if (PROJECT / 'dataset.zip').exists():
            with zipfile.ZipFile(PROJECT / 'dataset.zip') as z:
                z.extractall(ROOT)
        else:
            shutil.copytree(PROJECT / 'dataset', ROOT / 'dataset')
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    ROOT = PROJECT
    WORK = PROJECT / 'export_work'

SRC_MODELS = PROJECT / 'models' / SOURCE_VARIANT
SRC_RESULTS = PROJECT / 'results' / SOURCE_VARIANT
EXPORT = PROJECT / 'models' / 'export'
for d in (EXPORT / 'tflite_variants', EXPORT / 'test_vectors', WORK):
    d.mkdir(parents=True, exist_ok=True)
print('reading', SRC_MODELS, '| writing', EXPORT)

Mounted at /content/drive
reading /content/drive/MyDrive/seventx_prototype/models/crop_lr1e-4 | writing /content/drive/MyDrive/seventx_prototype/models/export


In [3]:
import tensorflow as tf
import keras
tf.get_logger().setLevel('ERROR')
print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| GPU:', bool(tf.config.list_physical_devices('GPU')))

TensorFlow 2.21.0 | Keras 3.13.2 | GPU: False


## 1. Preprocessing: one file for training, export and the apps

`preprocess.py` (the reference) and `preprocess.js` (the web port) are written to `models/export/`, and this notebook imports the Python one from there. Their source lives in the project at `notebooks/lib/`. If you change them, regenerate this notebook. Local checks already confirmed two things:
- `preprocess.fit_pad` is byte-identical to notebook 02's on all 1,123 images.
- The JS port gives the same crop boxes, with pixels within 2 levels.

In [4]:
PREPROCESS_PY = r'''"""Strawberry grader preprocessing: crop to the berry, fit inside 224x224, pad with the border colour.

This is the reference implementation. Training (notebook 02), export (notebook 03), the web demo
(preprocess.js) and the Android app must all do exactly this, or accuracy drops.

Model input: float32, shape (1, 224, 224, 3), RGB, values 0-255. NO normalisation (no /255):
EfficientNet-B0 and MobileNetV3 rescale inside the model. Output: P(damaged).
"""
import numpy as np
from PIL import Image, ImageOps

SIZE = 224          # model input size
MASK_SIZE = 256     # the berry mask is computed on a copy resized to 256x256
DIFF_MIN = 60       # |pixel - median border colour| summed over R, G, B must exceed this...
SAT_MIN = 70        # ...and the pixel must be coloured: HSV saturation (0-255) above this
LINE_FRAC = 0.8     # mask rows/columns more than 80% full are lines (e.g. a webcam edge), not berry
PCT = (1, 99)       # the box spans the 1st to 99th percentile of mask pixels
MARGIN = 0.15       # margin around the box, as a fraction of its size; then made square
MIN_PIXELS = 50     # fewer mask pixels than this: no crop


def berry_box(a, margin=MARGIN):
    """Box (x0, y0, x1, y1) around the berry or pile in RGB array `a`, or None if none is found."""
    h, w = a.shape[:2]
    img = Image.fromarray(a).resize((MASK_SIZE, MASK_SIZE))
    small = np.asarray(img, dtype=np.int16)
    sat = np.asarray(img.convert('HSV'), dtype=np.int16)[..., 1]
    bg = np.median(np.concatenate([small[0], small[-1], small[:, 0], small[:, -1]]), axis=0)
    mask = (np.abs(small - bg).sum(-1) > DIFF_MIN) & (sat > SAT_MIN)
    rows, cols = mask.mean(1) > LINE_FRAC, mask.mean(0) > LINE_FRAC
    mask[rows, :] = False
    mask[:, cols] = False
    ys, xs = np.where(mask)
    if len(ys) < MIN_PIXELS:
        return None
    (y0, y1) = np.percentile(ys, PCT) * h / MASK_SIZE
    (x0, x1) = np.percentile(xs, PCT) * w / MASK_SIZE
    side = max(x1 - x0, y1 - y0) * (1 + 2 * margin)
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    box = (max(0, cx - side / 2), max(0, cy - side / 2), min(w, cx + side / 2), min(h, cy + side / 2))
    return tuple(int(round(v)) for v in box)


def fit_pad_image(im, size=SIZE, crop=True):
    """Upright RGB PIL image -> (uint8 array size x size x 3, crop box or None)."""
    a = np.asarray(im)
    border = np.concatenate([a[0], a[-1], a[:, 0], a[:, -1]])
    bg = tuple(int(v) for v in np.median(border, axis=0))
    box = berry_box(a) if crop else None
    if box is not None:
        im = im.crop(box)
    im = im.copy()
    im.thumbnail((size, size), Image.LANCZOS)   # shrinks only, never enlarges
    canvas = Image.new('RGB', (size, size), bg)
    canvas.paste(im, ((size - im.width) // 2, (size - im.height) // 2))
    return np.asarray(canvas, dtype=np.uint8), box


def load_image(path, draft=False):
    """Open a photo upright (EXIF rotation applied). draft=True decodes big JPEGs at reduced size (training)."""
    with Image.open(path) as im:
        if draft:
            im.draft('RGB', (SIZE * 4, SIZE * 4))
        return ImageOps.exif_transpose(im).convert('RGB')


def fit_pad(path, size=SIZE, crop=True):
    """Exactly what training used (notebook 02): reduced-size JPEG decode, crop, fit, pad. uint8 array."""
    return fit_pad_image(load_image(path, draft=True), size, crop)[0]


def preprocess(image):
    """For apps and checks: a path or an upright RGB PIL image -> float32 model input (1, 224, 224, 3).
    Decodes at full size, as a phone or browser would."""
    im = load_image(image) if isinstance(image, (str, bytes)) or hasattr(image, '__fspath__') else image.convert('RGB')
    arr, _ = fit_pad_image(im)
    return arr.astype(np.float32)[None]
'''

PREPROCESS_JS = r'''/*
 * Strawberry grader preprocessing for the web demo: a port of preprocess.py.
 * Crop to the berry, fit inside 224x224 (shrink only), pad with the median border colour.
 *
 * Input:  an upright image as {width, height, data}, where data is RGBA (canvas ImageData) or RGB bytes.
 *         Browsers apply EXIF rotation when decoding, so draw the photo to a canvas and pass getImageData().
 * Output: {input: Float32Array(224*224*3) RGB 0-255 (NO /255), box: [x0,y0,x1,y1] or null, pixels: Uint8Array}
 *         Feed `input` to the model as shape [1, 224, 224, 3].
 *
 * Resampling copies Pillow's algorithm (antialiased convolution, fixed-point coefficients, two passes,
 * and thumbnail's pre-reduction), so the result matches the Python reference within a level or two.
 * Works in the browser (window.StrawberryPreprocess) and in Node (require).
 */
(function (root, factory) {
  if (typeof module === 'object' && module.exports) module.exports = factory();
  else root.StrawberryPreprocess = factory();
}(typeof self !== 'undefined' ? self : this, function () {
  'use strict';

  const P = { SIZE: 224, MASK_SIZE: 256, DIFF_MIN: 60, SAT_MIN: 70, LINE_FRAC: 0.8,
              PCT_LO: 1, PCT_HI: 99, MARGIN: 0.15, MIN_PIXELS: 50 };
  const PRECISION_BITS = 32 - 8 - 2;   // Pillow's fixed-point precision for 8-bit images

  function toRGB(img) {
    const w = img.width, h = img.height, d = img.data, ch = d.length / (w * h);
    const px = new Uint8Array(w * h * 3);
    for (let i = 0, j = 0; i < w * h; i++, j += ch) { px[3 * i] = d[j]; px[3 * i + 1] = d[j + 1]; px[3 * i + 2] = d[j + 2]; }
    return { w, h, px };
  }

  // ---- Pillow-compatible resampling -------------------------------------------------------------
  const FILTERS = {
    bicubic: { support: 2, fn: (x) => { const a = -0.5; x = Math.abs(x);
      if (x < 1) return ((a + 2) * x - (a + 3)) * x * x + 1;
      if (x < 2) return (((x - 5) * x + 8) * x - 4) * a; return 0; } },
    lanczos: { support: 3, fn: (x) => (x >= -3 && x < 3) ? sinc(x) * sinc(x / 3) : 0 },
  };
  function sinc(x) { if (x === 0) return 1; x *= Math.PI; return Math.sin(x) / x; }

  function coeffs(inSize, in0, in1, outSize, filter) {
    const scale = (in1 - in0) / outSize, filterscale = Math.max(scale, 1);
    const support = filter.support * filterscale, ksize = Math.ceil(support) * 2 + 1;
    const bounds = new Int32Array(outSize * 2), kk = new Float64Array(outSize * ksize);
    for (let xx = 0; xx < outSize; xx++) {
      const center = in0 + (xx + 0.5) * scale, ss = 1 / filterscale;
      let xmin = Math.trunc(center - support + 0.5); if (xmin < 0) xmin = 0;
      let xmax = Math.trunc(center + support + 0.5); if (xmax > inSize) xmax = inSize;
      xmax -= xmin;
      let ww = 0;
      for (let x = 0; x < xmax; x++) { const w = filter.fn((x + xmin - center + 0.5) * ss); kk[xx * ksize + x] = w; ww += w; }
      for (let x = 0; x < xmax; x++) {
        let k = ww !== 0 ? kk[xx * ksize + x] / ww : kk[xx * ksize + x];
        kk[xx * ksize + x] = Math.trunc(k * (1 << PRECISION_BITS) + (k < 0 ? -0.5 : 0.5));   // fixed point, as Pillow
      }
      bounds[2 * xx] = xmin; bounds[2 * xx + 1] = xmax;
    }
    return { ksize, bounds, kk };
  }
  const clip8 = (ss) => { const v = Math.floor(ss / (1 << PRECISION_BITS)); return v < 0 ? 0 : v > 255 ? 255 : v; };

  // Resize RGB image {w,h,px} to (ow, oh) from source box [x0,y0,x1,y1] (floats allowed), like Image.resize.
  function resample(src, ow, oh, filter, box) {
    box = box || [0, 0, src.w, src.h];
    const needH = ow !== src.w || box[0] !== 0 || box[2] !== src.w;
    const needV = oh !== src.h || box[1] !== 0 || box[3] !== src.h;
    const ch = coeffs(src.w, box[0], box[2], ow, filter), cv = coeffs(src.h, box[1], box[3], oh, filter);
    const half = 1 << (PRECISION_BITS - 1);
    let img = src;
    let vb = cv.bounds;
    if (needH) {
      const yFirst = cv.bounds[0], yLast = cv.bounds[2 * oh - 2] + cv.bounds[2 * oh - 1];
      vb = Int32Array.from(cv.bounds); for (let i = 0; i < oh; i++) vb[2 * i] -= yFirst;
      const th = yLast - yFirst, out = new Uint8Array(ow * th * 3);
      for (let y = 0; y < th; y++) {
        const row = (y + yFirst) * src.w;
        for (let xx = 0; xx < ow; xx++) {
          const xmin = ch.bounds[2 * xx], xmax = ch.bounds[2 * xx + 1], k = xx * ch.ksize;
          let s0 = half, s1 = half, s2 = half;
          for (let x = 0; x < xmax; x++) {
            const p = (row + x + xmin) * 3, c = ch.kk[k + x];
            s0 += src.px[p] * c; s1 += src.px[p + 1] * c; s2 += src.px[p + 2] * c;
          }
          const o = (y * ow + xx) * 3; out[o] = clip8(s0); out[o + 1] = clip8(s1); out[o + 2] = clip8(s2);
        }
      }
      img = { w: ow, h: th, px: out };
    }
    if (needV) {
      const out = new Uint8Array(img.w * oh * 3);
      for (let yy = 0; yy < oh; yy++) {
        const ymin = vb[2 * yy], ymax = vb[2 * yy + 1], k = yy * cv.ksize;
        for (let x = 0; x < img.w; x++) {
          let s0 = half, s1 = half, s2 = half;
          for (let y = 0; y < ymax; y++) {
            const p = ((y + ymin) * img.w + x) * 3, c = cv.kk[k + y];
            s0 += img.px[p] * c; s1 += img.px[p + 1] * c; s2 += img.px[p + 2] * c;
          }
          const o = (yy * img.w + x) * 3; out[o] = clip8(s0); out[o + 1] = clip8(s1); out[o + 2] = clip8(s2);
        }
      }
      img = { w: img.w, h: oh, px: out };
    }
    return img;
  }

  // Box-average downscale by integer factors (Image.reduce); edge blocks average what is there.
  function reduce(src, fx, fy) {
    const ow = Math.ceil(src.w / fx), oh = Math.ceil(src.h / fy), out = new Uint8Array(ow * oh * 3);
    for (let oy = 0; oy < oh; oy++) for (let ox = 0; ox < ow; ox++) {
      const y1 = Math.min((oy + 1) * fy, src.h), x1 = Math.min((ox + 1) * fx, src.w);
      let s0 = 0, s1 = 0, s2 = 0, n = 0;
      for (let y = oy * fy; y < y1; y++) for (let x = ox * fx; x < x1; x++) {
        const p = (y * src.w + x) * 3; s0 += src.px[p]; s1 += src.px[p + 1]; s2 += src.px[p + 2]; n++;
      }
      const o = (oy * ow + ox) * 3, a = Math.floor(n / 2);
      out[o] = Math.floor((s0 + a) / n); out[o + 1] = Math.floor((s1 + a) / n); out[o + 2] = Math.floor((s2 + a) / n);
    }
    return { w: ow, h: oh, px: out };
  }

  // Image.thumbnail((size, size), LANCZOS) on a non-JPEG image: shrink only, keep aspect, reducing_gap 2.
  function thumbnail(src, size) {
    if (size >= src.w && size >= src.h) return src;
    const aspect = src.w / src.h;
    let x = size, y = size;
    const roundAspect = (n, key) => Math.max([Math.floor(n), Math.ceil(n)].reduce((a, b) => key(b) < key(a) ? b : a), 1);
    if (x / y >= aspect) x = roundAspect(y * aspect, (n) => Math.abs(aspect - n / y));
    else y = roundAspect(x / aspect, (n) => n === 0 ? 0 : Math.abs(aspect - x / n));
    if (x === src.w && y === src.h) return src;
    const fx = Math.trunc(src.w / x / 2) || 1, fy = Math.trunc(src.h / y / 2) || 1;
    let img = src, box = [0, 0, src.w, src.h];
    if (fx > 1 || fy > 1) { img = reduce(src, fx, fy); box = [0, 0, src.w / fx, src.h / fy]; }
    return resample(img, x, y, FILTERS.lanczos, box);
  }

  // ---- the crop ----------------------------------------------------------------------------------
  function median(vals) {   // numpy median: mean of the two middle values for even counts
    const s = Float64Array.from(vals).sort(), n = s.length;
    return n % 2 ? s[(n - 1) / 2] : (s[n / 2 - 1] + s[n / 2]) / 2;
  }
  function borderMedian(img) {   // np.median(concatenate([a[0], a[-1], a[:,0], a[:,-1]]), axis=0)
    const out = [];
    for (let c = 0; c < 3; c++) {
      const v = [];
      for (let x = 0; x < img.w; x++) v.push(img.px[x * 3 + c]);
      for (let x = 0; x < img.w; x++) v.push(img.px[((img.h - 1) * img.w + x) * 3 + c]);
      for (let y = 0; y < img.h; y++) v.push(img.px[(y * img.w) * 3 + c]);
      for (let y = 0; y < img.h; y++) v.push(img.px[(y * img.w + img.w - 1) * 3 + c]);
      out.push(median(v));
    }
    return out;
  }
  function percentile(sorted, q) {   // numpy 'linear' percentile of a sorted array
    const i = (q / 100) * (sorted.length - 1), lo = Math.floor(i), hi = Math.ceil(i);
    return sorted[lo] + (sorted[hi] - sorted[lo]) * (i - lo);
  }
  function roundHalfEven(v) {   // Python's round()
    const f = Math.floor(v), d = v - f;
    if (d > 0.5) return f + 1; if (d < 0.5) return f; return f % 2 === 0 ? f : f + 1;
  }

  function berryBox(img) {
    const S = P.MASK_SIZE, small = resample(img, S, S, FILTERS.bicubic);
    const bg = borderMedian(small), mask = new Uint8Array(S * S);
    const rowCount = new Int32Array(S), colCount = new Int32Array(S);
    for (let i = 0; i < S * S; i++) {
      const r = small.px[3 * i], g = small.px[3 * i + 1], b = small.px[3 * i + 2];
      const diff = Math.abs(r - bg[0]) + Math.abs(g - bg[1]) + Math.abs(b - bg[2]);
      const mx = Math.max(r, g, b), mn = Math.min(r, g, b);
      const sat = mx === mn ? 0 : Math.trunc(Math.fround((mx - mn) / mx) * 255);   // PIL HSV saturation
      if (diff > P.DIFF_MIN && sat > P.SAT_MIN) { mask[i] = 1; rowCount[Math.floor(i / S)]++; colCount[i % S]++; }
    }
    const ys = [], xs = [];
    for (let y = 0; y < S; y++) {
      if (rowCount[y] / S > P.LINE_FRAC) continue;
      for (let x = 0; x < S; x++) if (mask[y * S + x] && !(colCount[x] / S > P.LINE_FRAC)) { ys.push(y); xs.push(x); }
    }
    if (ys.length < P.MIN_PIXELS) return null;
    ys.sort((a, b) => a - b); xs.sort((a, b) => a - b);
    const y0 = percentile(ys, P.PCT_LO) * img.h / S, y1 = percentile(ys, P.PCT_HI) * img.h / S;
    const x0 = percentile(xs, P.PCT_LO) * img.w / S, x1 = percentile(xs, P.PCT_HI) * img.w / S;
    const side = Math.max(x1 - x0, y1 - y0) * (1 + 2 * P.MARGIN), cx = (x0 + x1) / 2, cy = (y0 + y1) / 2;
    return [Math.max(0, cx - side / 2), Math.max(0, cy - side / 2), Math.min(img.w, cx + side / 2), Math.min(img.h, cy + side / 2)]
      .map(roundHalfEven);
  }

  function crop(img, [x0, y0, x1, y1]) {
    const w = x1 - x0, h = y1 - y0, out = new Uint8Array(w * h * 3);
    for (let y = 0; y < h; y++) out.set(img.px.subarray(((y + y0) * img.w + x0) * 3, ((y + y0) * img.w + x0 + w) * 3), y * w * 3);
    return { w, h, px: out };
  }

  function preprocess(imageData, options) {
    const doCrop = !options || options.crop !== false;
    let img = toRGB(imageData);
    const bg = borderMedian(img).map(Math.trunc);
    const box = doCrop ? berryBox(img) : null;
    if (box) img = crop(img, box);
    img = thumbnail(img, P.SIZE);
    const S = P.SIZE, pixels = new Uint8Array(S * S * 3);
    for (let i = 0; i < S * S; i++) { pixels[3 * i] = bg[0]; pixels[3 * i + 1] = bg[1]; pixels[3 * i + 2] = bg[2]; }
    const ox = Math.floor((S - img.w) / 2), oy = Math.floor((S - img.h) / 2);
    for (let y = 0; y < img.h; y++) pixels.set(img.px.subarray(y * img.w * 3, (y + 1) * img.w * 3), ((y + oy) * S + ox) * 3);
    return { input: Float32Array.from(pixels), box, pixels };
  }

  return { preprocess, berryBox, PARAMS: P };
}));
'''


(EXPORT / 'preprocess.py').write_text(PREPROCESS_PY, encoding='utf-8')
(EXPORT / 'preprocess.js').write_text(PREPROCESS_JS, encoding='utf-8')
import sys
sys.dont_write_bytecode = True                      # no __pycache__ in the export folder
shutil.rmtree(EXPORT / '__pycache__', ignore_errors=True)
spec = importlib.util.spec_from_file_location('preprocess', EXPORT / 'preprocess.py')
pp = importlib.util.module_from_spec(spec); spec.loader.exec_module(pp)
print('preprocess.py and preprocess.js written; crop margin', pp.MARGIN, '| input size', pp.SIZE)

preprocess.py and preprocess.js written; crop margin 0.15 | input size 224


## 2. Data: val, test and calibration images

In [5]:
with open(PROJECT / 'data' / 'manifest.csv', newline='') as f:
    M = list(csv.DictReader(f))
assert len(M) == EXPECTED_MANIFEST_ROWS, f'manifest has {len(M)} rows, expected {EXPECTED_MANIFEST_ROWS}'
for r in M:
    r['y'] = 1 if r['label'] == 'damaged' else 0
    r['multi_berry'] = int(r['multi_berry'])

VAL = [i for i, r in enumerate(M) if r['split'] == 'val']
TEST = [i for i, r in enumerate(M) if r['split'] == 'test']
REP = sorted(random.Random(0).sample([i for i, r in enumerate(M) if r['split'] == 'train'], N_REPRESENTATIVE))
t0 = time.time()
ARR = {i: pp.fit_pad(ROOT / M[i]['path']) for i in VAL + TEST + REP}   # exactly as in training
print(f'{len(VAL)} val, {len(TEST)} test, {len(REP)} calibration images preprocessed in {time.time() - t0:.0f} s')
assert (len(VAL), len(TEST)) == (165, 161)

def batch(ix):
    return np.stack([ARR[i] for i in ix]).astype(np.float32)

165 val, 161 test, 200 calibration images preprocessed in 27 s


## 3. Which trained model to export

For each model, the E1 seed with the best **val** accuracy (from notebook 02's `runs.csv`), tie-broken by the lowest val loss.

In [6]:
with open(SRC_RESULTS / 'runs.csv', newline='') as f:
    RUNS = list(csv.DictReader(f))
SEED = {}
for name in MODELS:
    cands = []
    for r in RUNS:
        if (r['exp'], r['model'], r['eval_on'], r['subset'], r['threshold']) == ('E1', name, 'val', 'all', '0.5'):
            hist = json.loads((SRC_RESULTS / 'history' / f"E1_{name}_s{r['seed']}.json").read_text())
            cands.append((float(r['acc']), -hist['best_val_loss'], int(r['seed'])))
    cands.sort(reverse=True)
    SEED[name] = cands[0][2]
    print(name, 'seeds (val acc, val loss):', [(s, round(a, 4), round(-l, 4)) for a, l, s in cands], '-> seed', SEED[name])

effb0 seeds (val acc, val loss): [(42, 0.9212, 0.1828), (43, 0.9212, 0.1972), (44, 0.9212, 0.1996)] -> seed 42


## 4. Parity check: does the exported preprocessing match training?

Predict val and test with the float32 Keras model on images from `preprocess.py`, and compare with notebook 02's saved predictions.

In [7]:
def read_probs(path):
    with open(path, newline='') as f:
        return {r['path']: float(r['prob_damaged']) for r in csv.DictReader(f)}

KERAS, P_KERAS = {}, {}
for name in MODELS:
    tag = f'E1_{name}_s{SEED[name]}'
    KERAS[name] = keras.models.load_model(str(SRC_MODELS / f'{tag}.keras'))
    worst = 0.0
    for part, ix, f in (('val', VAL, f'{tag}_val.csv'), ('test', TEST, f'{tag}.csv')):
        p = KERAS[name].predict(batch(ix), verbose=0).ravel()
        P_KERAS[(name, part)] = p
        saved = read_probs(SRC_RESULTS / 'preds' / f)
        worst = max(worst, max(abs(p[k] - saved[M[i]['path']]) for k, i in enumerate(ix)))
    print(f'{name}: max |dp| vs notebook 02 = {worst:.2e}')
    assert worst < 1e-3, 'preprocessing or model differs from training'
print('parity OK: export preprocessing = training preprocessing')

effb0: max |dp| vs notebook 02 = 5.30e-06
parity OK: export preprocessing = training preprocessing


## 5. TFLite conversion

Four variants. Input and output stay float32 in all of them, so the app contract never changes.

| Variant | How |
| --- | --- |
| `float32` | no optimisation (reference) |
| `float16` | weights stored as float16 |
| `dynamic_int8` | weights int8, activations float |
| `int8` | weights and activations int8, calibrated on 200 train images; ops without int8 kernels stay float |

In [8]:
VARIANTS = ['float32', 'float16', 'dynamic_int8', 'int8']

def representative():
    for i in REP:
        yield [ARR[i][None].astype(np.float32)]

def convert(saved_model_dir, variant):
    c = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
    if variant == 'float16':
        c.optimizations = [tf.lite.Optimize.DEFAULT]
        c.target_spec.supported_types = [tf.float16]
    elif variant == 'dynamic_int8':
        c.optimizations = [tf.lite.Optimize.DEFAULT]
    elif variant == 'int8':
        c.optimizations = [tf.lite.Optimize.DEFAULT]
        c.representative_dataset = representative
    return c.convert()

TFLITE = {}
for name in MODELS:
    sm = WORK / f'savedmodel_{name}'
    if sm.exists():
        shutil.rmtree(sm)
    KERAS[name].export(str(sm))
    for v in VARIANTS:
        path = EXPORT / 'tflite_variants' / f'{name}_{v}.tflite'
        try:
            t0 = time.time()
            path.write_bytes(convert(sm, v))
            TFLITE[(name, v)] = path
            print(f'{name:6s} {v:13s} {path.stat().st_size / 1e6:6.2f} MB  ({time.time() - t0:.0f} s)')
        except Exception as e:
            print(f'{name:6s} {v:13s} FAILED: {type(e).__name__}: {str(e)[:200]}')

Saved artifact at '/content/export_work/savedmodel_effb0'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134814391239504: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  134814391239312: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  134814783912080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134814783907088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134814774593104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134814783911888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134814774592912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134814774592144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134814774591760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134814774592528: TensorSpec(shape=(), dtype=tf.resou

## 6. Accuracy and speed of every variant

`tf.lite.Interpreter` on the CPU, one image at a time, as an app does. Speed is the median of 50 runs after 5 warm-up runs. Free Colab has only 2 CPU cores, so treat ms/image as a rough reference; phone timing comes from the apps.

In [9]:
def tflite_predict(path, x):
    it = tf.lite.Interpreter(model_path=str(path), num_threads=4)
    it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    probs = np.empty(len(x), dtype=np.float32)
    for k in range(len(x)):
        it.set_tensor(inp['index'], x[k:k + 1])
        it.invoke()
        probs[k] = it.get_tensor(out['index']).ravel()[0]
    times = []
    for k in range(TIMING_WARMUP + TIMING_RUNS):
        it.set_tensor(inp['index'], x[k % len(x):k % len(x) + 1])
        t0 = time.perf_counter(); it.invoke(); times.append(time.perf_counter() - t0)
    return probs, 1000 * float(np.median(times[TIMING_WARMUP:]))

def scores(y, p, thr):
    pred = (p >= thr).astype(int)
    tp, fn = int(((pred == 1) & (y == 1)).sum()), int(((pred == 0) & (y == 1)).sum())
    tn, fp = int(((pred == 0) & (y == 0)).sum()), int(((pred == 1) & (y == 0)).sum())
    div = lambda a, b: a / b if b else float('nan')
    return dict(acc=div(tp + tn, len(y)), damaged_recall=div(tp, tp + fn), good_precision=div(tn, tn + fn),
                damaged_precision=div(tp, tp + fp), tp=tp, fn=fn, tn=tn, fp=fp)

def recall_threshold(y, p, target=RECALL_TARGET):
    # Largest threshold that still gives damaged recall >= target (not capped at 0.5 here).
    d = np.sort(p[y == 1])
    return float(d[int(math.floor((1 - target) * len(d)))])

Y = {'val': np.array([M[i]['y'] for i in VAL]), 'test': np.array([M[i]['y'] for i in TEST])}
X = {'val': batch(VAL), 'test': batch(TEST)}
ROWS, P_TFLITE = [], {}
for (name, v), path in TFLITE.items():
    pv, ms = tflite_predict(path, X['val'])
    pt, _ = tflite_predict(path, X['test'])
    assert len(pv) == 165 and len(pt) == 161
    P_TFLITE[(name, v, 'val')], P_TFLITE[(name, v, 'test')] = pv, pt
    if v == 'float32':
        d = float(np.abs(pv - P_KERAS[(name, 'val')]).max())
        print(f'{name} float32 TFLite vs Keras: max |dp| = {d:.2e}')
        assert d < 1e-3, 'float32 TFLite does not match Keras'
    thr = recall_threshold(Y['val'], pv)
    row = dict(model=name, seed=SEED[name], variant=v, mb=round(path.stat().st_size / 1e6, 2), ms=round(ms, 1), threshold=round(thr, 4))
    for part, p in (('val', pv), ('test', pt)):
        for tname, t in (('05', 0.5), ('thr', thr)):
            for k, val in scores(Y[part], p, t).items():
                row[f'{part}_{tname}_{k}'] = val
    ROWS.append(row)
    print(f"{name:6s} {v:13s} {row['mb']:6.2f} MB {ms:7.1f} ms/img | val acc {row['val_05_acc']:.3f}, damaged recall {row['val_05_damaged_recall']:.3f}")

with open(SRC_RESULTS / 'export_runs.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, list(ROWS[0])); w.writeheader(); w.writerows(ROWS)

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


effb0 float32 TFLite vs Keras: max |dp| = 2.71e-05
effb0  float32        16.06 MB    30.1 ms/img | val acc 0.921, damaged recall 0.887
effb0  float16         8.12 MB    29.9 ms/img | val acc 0.921, damaged recall 0.887
effb0  dynamic_int8    4.56 MB   116.5 ms/img | val acc 0.933, damaged recall 0.950
effb0  int8            4.93 MB    29.1 ms/img | val acc 0.806, damaged recall 1.000


## 7. Choose what ships (val only)

**`SHIP_VARIANT` (float16)** if its val accuracy **and** val damaged recall at threshold 0.5 are within `TOLERANCE` of float32. Otherwise, the rule: among the variants that pass, the **fastest**; ties go to the smaller file. (Why it's pinned: free Colab's CPU timings are too noisy to rank variants that run at the same speed. Float16 and float32 swapped places between two runs, and the second run shipped float32 at 16 MB.) If none passes, float16. (The first run used "smallest" and picked dynamic int8: 4.6 MB but 4× slower than float16 at 92 ms, and with shifted predictions. Float16 is 8.1 MB and gives the same answers as float32.) If the model is slower than `SPEED_LIMIT_MS`, the notebook warns; with `'mnv3s'` added to `MODELS` it would ship MobileNet instead. The threshold is the chosen variant's val-tuned one.

In [10]:
def choose_variant(name):
    rows = {r['variant']: r for r in ROWS if r['model'] == name}
    ref = rows['float32']
    ok = [r for r in rows.values()
          if r['val_05_acc'] >= ref['val_05_acc'] - TOLERANCE and r['val_05_damaged_recall'] >= ref['val_05_damaged_recall'] - TOLERANCE]
    if SHIP_VARIANT and SHIP_VARIANT in [r['variant'] for r in ok]:
        return rows[SHIP_VARIANT]
    if SHIP_VARIANT:
        print(f'WARNING: {SHIP_VARIANT} is not within {TOLERANCE * 100:.0f} point(s) of float32 on val; using the rule instead')
    return min(ok, key=lambda r: (r['ms'], r['mb'])) if ok else rows.get('float16', ref)

CHOICE = {name: choose_variant(name) for name in MODELS}
ship = CHOICE[SHIP_MODEL]
why = (f'{SHIP_MODEL} {ship["variant"]}: ' + ('SHIP_VARIANT, checked within ' if ship['variant'] == SHIP_VARIANT else 'fastest variant within ')
       + f'{TOLERANCE * 100:.0f} point(s) of float32 on val')
if ship['ms'] > SPEED_LIMIT_MS:
    fallback = next((n for n in MODELS if n != SHIP_MODEL), None)
    if fallback:
        why = f'{SHIP_MODEL} too slow ({ship["ms"]} ms > {SPEED_LIMIT_MS} ms): shipping {fallback} {CHOICE[fallback]["variant"]}'
        ship = CHOICE[fallback]
    else:
        print(f'WARNING: {SHIP_MODEL} takes {ship["ms"]} ms/image on this CPU (limit {SPEED_LIMIT_MS}). Shipping it anyway; '
              "measure on the phone. To compare with MobileNet, add 'mnv3s' to MODELS and rerun.")
SHIPPED = ship
THRESHOLD = ship['threshold']
shutil.copy(TFLITE[(ship['model'], ship['variant'])], EXPORT / 'strawberry_2class.tflite')
print('SHIPPED:', why)
print(f"  {ship['mb']} MB, {ship['ms']} ms/image (Colab CPU), threshold {THRESHOLD:.3f}")
print(f"  val @0.5: acc {ship['val_05_acc']:.3f}, damaged recall {ship['val_05_damaged_recall']:.3f}")
print(f"  val @{THRESHOLD:.3f}: acc {ship['val_thr_acc']:.3f}, damaged recall {ship['val_thr_damaged_recall']:.3f}, good precision {ship['val_thr_good_precision']:.3f}")

SHIPPED: effb0 float16: SHIP_VARIANT, checked within 1 point(s) of float32 on val
  8.12 MB, 29.9 ms/image (Colab CPU), threshold 0.261
  val @0.5: acc 0.921, damaged recall 0.887
  val @0.261: acc 0.927, damaged recall 0.950, good precision 0.951


In [11]:
# Test scores of the shipped model: printed AFTER the choice, for reporting only.
s = SHIPPED
print(f"TEST (161 images) @0.5:   acc {s['test_05_acc']:.3f}, damaged recall {s['test_05_damaged_recall']:.3f}, good precision {s['test_05_good_precision']:.3f}")
print(f"TEST @{THRESHOLD:.3f}: acc {s['test_thr_acc']:.3f}, damaged recall {s['test_thr_damaged_recall']:.3f}, good precision {s['test_thr_good_precision']:.3f}")
print(f"confusion @{THRESHOLD:.3f}: TN {s['test_thr_tn']}  FP {s['test_thr_fp']}  FN {s['test_thr_fn']}  TP {s['test_thr_tp']}")

TEST (161 images) @0.5:   acc 0.981, damaged recall 0.964, good precision 0.963
TEST @0.261: acc 0.969, damaged recall 0.964, good precision 0.962
confusion @0.261: TN 75  FP 2  FN 3  TP 81


## 8. Test vectors for the app ports

Ten **Mendeley** photos (CC BY 4.0; Mahendra's GPL photos must not be redistributed): fresh singles, fresh piles, a crate, a dark mouldy berry, a mashed berry, rotten piles and a rotten single. For each one: the crop box, the 224×224 output's mean and std, and P(damaged) from the float32 model and the shipped `.tflite`, all on a **full-size decode**, as a phone or browser does. A port passes if its boxes agree within 1% of the photo's longer side and its P is within 0.02.

In [12]:
import re
num = lambda r: int(re.search(r'\((\d+)\)', r['path']).group(1))
MEN = sorted((r for r in M if r['source'] == 'mendeley'), key=lambda r: (r['label'], num(r)))
want = [  # (description, condition, how many)
    ('fresh single', lambda r: r['label'] == 'good' and r['multi_berry'] == 0 and not 173 <= num(r) <= 177, 2),
    ('fresh pile', lambda r: r['label'] == 'good' and r['multi_berry'] == 1 and not 173 <= num(r) <= 177, 2),
    ('fresh crate', lambda r: r['label'] == 'good' and 173 <= num(r) <= 177, 1),
    ('rotten dark mouldy', lambda r: r['label'] == 'damaged' and 169 <= num(r) <= 197, 1),
    ('rotten mashed', lambda r: r['label'] == 'damaged' and 151 <= num(r) <= 166, 1),
    ('rotten pile', lambda r: r['label'] == 'damaged' and r['multi_berry'] == 1, 2),
    ('rotten single', lambda r: r['label'] == 'damaged' and r['multi_berry'] == 0 and num(r) < 151, 1),
]
rng = random.Random(1)
ship_keras = KERAS[SHIPPED['model']]
it = tf.lite.Interpreter(model_path=str(EXPORT / 'strawberry_2class.tflite')); it.allocate_tensors()
inp, outp = it.get_input_details()[0], it.get_output_details()[0]
VECTORS = []
for desc, cond, n in want:
    for r in rng.sample([r for r in MEN if cond(r)], n):
        src = ROOT / r['path']
        name = src.name.replace(' ', '_').replace('(', '').replace(')', '')
        shutil.copy(src, EXPORT / 'test_vectors' / name)
        im = pp.load_image(src)                       # full-size decode, upright
        arr, box = pp.fit_pad_image(im)
        x = arr.astype(np.float32)[None]
        p_keras = float(ship_keras.predict(x, verbose=0).ravel()[0])
        it.set_tensor(inp['index'], x); it.invoke()
        p_tfl = float(it.get_tensor(outp['index']).ravel()[0])
        p_train = float(ship_keras.predict(pp.fit_pad(src).astype(np.float32)[None], verbose=0).ravel()[0])
        VECTORS.append(dict(file=name, description=desc, label=r['label'], width=im.width, height=im.height,
                            box=list(box) if box else None, mean=round(float(arr.mean()), 3), std=round(float(arr.std()), 3),
                            p_damaged_float32=round(p_keras, 5), p_damaged_tflite=round(p_tfl, 5)))
        print(f'{desc:20s} {name:28s} box {box}  P float32 {p_keras:.3f} | tflite {p_tfl:.3f} | training-style decode {p_train:.3f}')
(EXPORT / 'test_vectors' / 'test_vectors.json').write_text(json.dumps(dict(
    pass_criteria=dict(box_tolerance='1% of the longer image side', p_tolerance=0.02),
    note='Decode at full size and apply EXIF rotation; feed RGB 0-255 float32, no /255.',
    vectors=VECTORS), indent=1))
print(len(VECTORS), 'test vectors written')

fresh single         FreshStrawberry_35.jpg       box (0, 0, 1373, 2380)  P float32 0.001 | tflite 0.001 | training-style decode 0.001
fresh single         FreshStrawberry_17.jpg       box (0, 0, 1348, 1929)  P float32 0.000 | tflite 0.000 | training-style decode 0.000
fresh pile           FreshStrawberry_104.jpg      box (0, 0, 3396, 2926)  P float32 0.000 | tflite 0.000 | training-style decode 0.000
fresh pile           FreshStrawberry_95.jpg       box (0, 0, 2586, 2826)  P float32 0.000 | tflite 0.000 | training-style decode 0.000
fresh crate          FreshStrawberry_176.jpg      box (0, 0, 3120, 4160)  P float32 0.006 | tflite 0.006 | training-style decode 0.006
rotten dark mouldy   RottenStrawberry_193.jpg     box (218, 470, 2653, 2905)  P float32 1.000 | tflite 1.000 | training-style decode 1.000
rotten mashed        RottenStrawberry_165.jpg     box (617, 627, 2034, 2044)  P float32 1.000 | tflite 1.000 | training-style decode 1.000
rotten pile          RottenStrawberry_145.jpg  

## 9. `labels.txt` and `metadata.json`

In [13]:
(EXPORT / 'labels.txt').write_text('good\ndamaged\n')

def run_score(model, exp):
    for r in RUNS:
        if (r['exp'], r['model'], r['eval_on'], r['subset'], r['threshold'], r['seed']) == (exp, model, 'test', 'all', '0.5', '42'):
            return round(float(r['acc']), 4)
    return None

s = SHIPPED
META = {
    'name': 'strawberry_2class', 'version': dt.date.today().isoformat(),
    'model': {'architecture': {'effb0': 'EfficientNet-B0', 'mnv3s': 'MobileNetV3-Small'}[s['model']],
              'trained_run': f"E1_{s['model']}_s{s['seed']} ({SOURCE_VARIANT})", 'tflite_variant': s['variant'],
              'file': 'strawberry_2class.tflite', 'size_mb': s['mb'], 'tf': tf.__version__, 'keras': keras.__version__},
    'input': {'shape': [1, pp.SIZE, pp.SIZE, 3], 'dtype': 'float32', 'color': 'RGB', 'range': [0, 255],
              'normalisation': 'none (do NOT divide by 255; the model rescales internally)',
              'preprocessing': 'preprocess.py / preprocess.js: EXIF-upright, crop to berry, fit inside 224x224 (shrink only), pad with median border colour'},
    'output': {'shape': [1, 1], 'meaning': 'P(damaged)'},
    'labels': ['good', 'damaged'],
    'threshold': {'value': THRESHOLD, 'rule': f'damaged if P >= threshold; largest threshold with >= {RECALL_TARGET:.0%} damaged recall on the validation set (165 images)'},
    'crop': {'mask_size': pp.MASK_SIZE, 'diff_min': pp.DIFF_MIN, 'sat_min': pp.SAT_MIN, 'line_frac': pp.LINE_FRAC,
             'percentiles': list(pp.PCT), 'margin': pp.MARGIN, 'min_pixels': pp.MIN_PIXELS, 'size': pp.SIZE},
    'scores': {
        'val_at_0.5': {k: round(s[f'val_05_{k}'], 4) for k in ('acc', 'damaged_recall', 'good_precision')},
        'val_at_threshold': {k: round(s[f'val_thr_{k}'], 4) for k in ('acc', 'damaged_recall', 'good_precision')},
        'test_at_0.5': {k: round(s[f'test_05_{k}'], 4) for k in ('acc', 'damaged_recall', 'good_precision')},
        'test_at_threshold': {k: round(s[f'test_thr_{k}'], 4) for k in ('acc', 'damaged_recall', 'good_precision')},
        'cross_dataset_float32_seed42': {'E2a_train_mendeley_test_mahendra_acc': run_score(s['model'], 'E2a'),
                                         'E2b_train_mahendra_test_mendeley_acc': run_score(s['model'], 'E2b')},
        'warning': 'Accuracy drops on photos from a new camera or setting (cross-dataset 63-75%). Collect and fine-tune on local photos before relying on it.',
    },
    'speed_ms_colab_cpu': s['ms'],
    'runtimes': {'android': 'TensorFlow Lite / LiteRT Interpreter, strawberry_2class.tflite',
                 'web': 'the same strawberry_2class.tflite with the LiteRT / TFLite web (WebAssembly) runtime; preprocess.js for the crop'},
    'attribution': [
        'Fresh and Rotten Fruits Dataset, Mendeley Data, DOI 10.17632/bdd69gyhv8.1, CC BY 4.0',
        'Mahendra et al. 2018 strawberry dataset (GPL), https://github.com/okamahendra123/dataset_strawberry',
    ],
}
(EXPORT / 'metadata.json').write_text(json.dumps(META, indent=1))
for key in ('model', 'input', 'output', 'labels', 'threshold', 'crop', 'scores', 'attribution'):
    assert key in json.loads((EXPORT / 'metadata.json').read_text())
print(json.dumps(META, indent=1)[:1500], '...')

{
 "name": "strawberry_2class",
 "version": "2026-11-27",
 "model": {
  "architecture": "EfficientNet-B0",
  "trained_run": "E1_effb0_s42 (crop_lr1e-4)",
  "tflite_variant": "float16",
  "file": "strawberry_2class.tflite",
  "size_mb": 8.12,
  "tf": "2.21.0",
  "keras": "3.13.2"
 },
 "input": {
  "shape": [
   1,
   224,
   224,
   3
  ],
  "dtype": "float32",
  "color": "RGB",
  "range": [
   0,
   255
  ],
  "normalisation": "none (do NOT divide by 255; the model rescales internally)",
  "preprocessing": "preprocess.py / preprocess.js: EXIF-upright, crop to berry, fit inside 224x224 (shrink only), pad with median border colour"
 },
 "output": {
  "shape": [
   1,
   1
  ],
  "meaning": "P(damaged)"
 },
 "labels": [
  "good",
  "damaged"
 ],
 "threshold": {
  "value": 0.2611,
  "rule": "damaged if P >= threshold; largest threshold with >= 95% damaged recall on the validation set (165 images)"
 },
 "crop": {
  "mask_size": 256,
  "diff_min": 60,
  "sat_min": 70,
  "line_frac": 0.8,
  "

## 10. Web demo check

The web demo runs the **same** `strawberry_2class.tflite` in the browser with Google's LiteRT/TFLite web runtime (WebAssembly), so both demos share one model file. No TF.js conversion is needed; it failed on Colab's Python 3.13 anyway.

This cell checks the part the web demo adds, `preprocess.js`. Node runs it on the test-vector photos, then its 224×224 output goes through the shipped `.tflite` here and is compared with the Python reference:
- crop boxes within 1% of the photo's longer side;
- P(damaged) within 0.02.

Running the browser runtime itself is checked in the browser (see `export.md`).

In [14]:
def sh(cmd, timeout=900):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return r.returncode, (r.stdout + r.stderr)[-3000:]

WEB_CHECK = 'skipped (RUN_WEB_CHECK = False)'
if RUN_WEB_CHECK:
    if shutil.which('node') is None:
        WEB_CHECK = 'skipped: node not found on this machine'
    else:
        chk = WORK / 'web_check'
        chk.mkdir(exist_ok=True)
        cases = []
        for v in VECTORS:   # full-size decode in Python, so only preprocess.js is being tested
            im = pp.load_image(EXPORT / 'test_vectors' / v['file'])
            raw = chk / (v['file'] + '.rgb')
            raw.write_bytes(np.asarray(im).tobytes())
            cases.append(dict(file=v['file'], w=im.width, h=im.height, raw=str(raw)))
        (chk / 'cases.json').write_text(json.dumps(cases))
        (chk / 'run.js').write_text(r'''
const fs = require('fs'); const pp = require(process.argv[2]);
const out = [];
for (const c of JSON.parse(fs.readFileSync(process.argv[3]))) {
  const r = pp.preprocess({width: c.w, height: c.h, data: fs.readFileSync(c.raw)});
  fs.writeFileSync(c.raw + '.js.bin', Buffer.from(r.pixels));
  out.push({file: c.file, box: r.box});
}
console.log(JSON.stringify(out));
''')
        code_, log = sh(f"node {chk / 'run.js'} {EXPORT / 'preprocess.js'} {chk / 'cases.json'}")
        if code_ != 0:
            WEB_CHECK = 'node failed: ' + log[-500:]
        else:
            res = {o['file']: o for o in json.loads(log.strip().splitlines()[-1])}
            it = tf.lite.Interpreter(model_path=str(EXPORT / 'strawberry_2class.tflite')); it.allocate_tensors()
            inp, outp = it.get_input_details()[0], it.get_output_details()[0]
            worst_p, worst_box = 0.0, 0.0
            for v, c in zip(VECTORS, cases):
                px = np.frombuffer(Path(c['raw'] + '.js.bin').read_bytes(), dtype=np.uint8).reshape(pp.SIZE, pp.SIZE, 3)
                it.set_tensor(inp['index'], px.astype(np.float32)[None]); it.invoke()
                p = float(it.get_tensor(outp['index']).ravel()[0])
                worst_p = max(worst_p, abs(p - v['p_damaged_tflite']))
                jb = res[v['file']]['box']
                if v['box'] and jb:
                    worst_box = max(worst_box, max(abs(a - b) for a, b in zip(v['box'], jb)) / max(v['width'], v['height']))
                elif v['box'] != jb:
                    worst_box = 1.0
            WEB_CHECK = f'preprocess.js + shipped .tflite: max |dp| {worst_p:.4f} (limit 0.02), max box diff {100 * worst_box:.2f}% of side (limit 1%)'
            assert worst_p < 0.02 and worst_box <= 0.01, WEB_CHECK
        shutil.rmtree(chk, ignore_errors=True)   # the raw files are large
print('Web check:', WEB_CHECK)

Web check: preprocess.js + shipped .tflite: max |dp| 0.0000 (limit 0.02), max box diff 0.00% of side (limit 1%)


## 11. `export.md`

In [15]:
pct = lambda v: f'{100 * v:.1f}'
L = ['# Export results', '',
     f'Generated by notebook 03 on {dt.date.today().isoformat()}. Choices use **val** only; test is reported after the choice.', '',
     f"**Shipped:** {SHIPPED['model']} `{SHIPPED['variant']}` from run E1_{SHIPPED['model']}_s{SHIPPED['seed']} "
     f"as `strawberry_2class.tflite`: {SHIPPED['mb']} MB, {SHIPPED['ms']} ms/image on the Colab CPU, threshold **{THRESHOLD:.3f}**.", '',
     '| Model | Variant | MB | ms/img | val acc | val damaged recall | Δ acc vs float32 | test acc | test damaged recall |',
     '| --- | --- | --- | --- | --- | --- | --- | --- | --- |']
for r in ROWS:
    ref = next(x for x in ROWS if x['model'] == r['model'] and x['variant'] == 'float32')
    L.append(f"| {r['model']} | {r['variant']} | {r['mb']} | {r['ms']} | {pct(r['val_05_acc'])} | {pct(r['val_05_damaged_recall'])} | "
             f"{100 * (r['val_05_acc'] - ref['val_05_acc']):+.1f} | {pct(r['test_05_acc'])} | {pct(r['test_05_damaged_recall'])} |")
s = SHIPPED
L += ['', 'Threshold 0.5 in the table. The shipped model at its val-tuned threshold:', '',
      '| | threshold | accuracy | damaged recall | good precision |', '| --- | --- | --- | --- | --- |',
      f"| val | {THRESHOLD:.3f} | {pct(s['val_thr_acc'])} | {pct(s['val_thr_damaged_recall'])} | {pct(s['val_thr_good_precision'])} |",
      f"| test | {THRESHOLD:.3f} | {pct(s['test_thr_acc'])} | {pct(s['test_thr_damaged_recall'])} | {pct(s['test_thr_good_precision'])} |", '',
      '## Checks', '',
      '- Preprocessing parity with training: max |Δp| < 1e-3 (asserted).',
      '- float32 TFLite vs Keras: max |Δp| < 1e-3 (asserted).',
      f'- Web check: {WEB_CHECK}.', '',
      'Both demos use `strawberry_2class.tflite`. In the browser, load it with the LiteRT / TFLite web runtime and check the test vectors there once.', '',
      'Speed is a Colab-CPU reference (2 cores); measure on the phone and in the browser for real numbers.']
(SRC_RESULTS / 'export.md').write_text('\n'.join(L) + '\n', encoding='utf-8')
print('\n'.join(L))
print('\nFiles in', EXPORT, ':', sorted(p.name for p in EXPORT.iterdir()))
print('Next: download models/export/ from Drive into the project folder for the app builders.')

# Export results

Generated by notebook 03 on 2026-11-27. Choices use **val** only; test is reported after the choice.

**Shipped:** effb0 `float16` from run E1_effb0_s42 as `strawberry_2class.tflite`: 8.12 MB, 29.9 ms/image on the Colab CPU, threshold **0.261**.

| Model | Variant | MB | ms/img | val acc | val damaged recall | Δ acc vs float32 | test acc | test damaged recall |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| effb0 | float32 | 16.06 | 30.1 | 92.1 | 88.8 | +0.0 | 98.1 | 96.4 |
| effb0 | float16 | 8.12 | 29.9 | 92.1 | 88.8 | +0.0 | 98.1 | 96.4 |
| effb0 | dynamic_int8 | 4.56 | 116.5 | 93.3 | 95.0 | +1.2 | 96.9 | 96.4 |
| effb0 | int8 | 4.93 | 29.1 | 80.6 | 100.0 | -11.5 | 88.2 | 98.8 |

Threshold 0.5 in the table. The shipped model at its val-tuned threshold:

| | threshold | accuracy | damaged recall | good precision |
| --- | --- | --- | --- | --- |
| val | 0.261 | 92.7 | 95.0 | 95.1 |
| test | 0.261 | 96.9 | 96.4 | 96.2 |

## Checks

- Preprocessing parity w